# Inference with a Hub model

Load a MOUSE model from the Hugging Face Hub and run a single cached forward step.
Expand the single step into a full environment loop for real rollouts.

In [ ]:
import os

import torch
from tensordict import TensorDict

from mouse_core import load_model

MODEL_ID = os.environ.get("MODEL_ID", "")  # e.g. "your-org/your-mouse-model"
assert MODEL_ID, "Set MODEL_ID to a MOUSE model on the Hub (e.g. your-org/your-model)."

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model(MODEL_ID).eval().to(device)

## Single cached forward step

The `step_stream` fields are the internal `TensorDict` keys produced by `DatasetStore`.

In [ ]:
step_stream = TensorDict(
    {
        "action": torch.zeros(1, 1, dtype=torch.long),
        "reward": torch.zeros(1, 1, dtype=torch.float32),
        "done": torch.zeros(1, 1, dtype=torch.long),
        "time": torch.zeros(1, 1, dtype=torch.long),
        "obs_discrete": torch.zeros(1, 1, dtype=torch.long),
    },
    batch_size=(1, 1),
)

with torch.no_grad():
    out, cache = model(step_stream.to(device), cache=None, use_cache=True)

action = model.get_action(out, temperature=0.0)
print(f"action={action.tolist()} cache_keys={list(cache.keys()) if cache else None}")